# Extra checks C1 + C3 - seed-averaged reference DTF and Diebold-Mariano test (free Colab, about 1 hour)

**C1 - fairness check.** TSO-DTF averages several models; the reference DTF is one model. Here the reference DTF is re-trained
with **exactly its own tuned settings** (read from `baseline_rerun_params.csv` or, if missing, from `tuning_log.csv`) but averaged over **5 random seeds**, and TSO-DTF is
compared with this stronger reference.

**C3 - significance per day.** Diebold-Mariano tests (with the Harvey-Leybourne-Newbold small-sample correction) on the 62 daily
losses, plus effect sizes of the per-product improvement.

Nothing in your existing results is changed; new files: `reference5_predictions.csv.gz`, `extra_checks_*.csv`.

**How to run:** Runtime -> Run all (about 1 hour). If Colab disconnects, Run all again - finished stores are skipped.
At the end `extra_checks_results.zip` downloads - send it to the chat.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    get_ipython().system("pip install -q lightgbm pyarrow")

In [ ]:
import os, gc, glob, shutil, time
import numpy as np, pandas as pd, lightgbm as lgb
from scipy import stats
import warnings; warnings.filterwarnings("ignore")

DATA_DIR = os.environ.get("M5_DATA_DIR", "/content/drive/MyDrive/m5_data")
OUT_DIR = os.environ.get("M5_OUT_DIR", "/content/drive/MyDrive/m5_results")
WORK_DIR = os.environ.get("M5_WORK_DIR", "/content/m5_work_c1")
STEP_DIR = f"{OUT_DIR}/extra_checks_work"
MERGED_DIR, FEATS_DIR = f"{WORK_DIR}/merged", f"{WORK_DIR}/features"
for d in [OUT_DIR, STEP_DIR, MERGED_DIR, FEATS_DIR]:
    os.makedirs(d, exist_ok=True)
if not os.path.exists(f"{DATA_DIR}/sell_prices.csv"):
    raise FileNotFoundError("Put the M5 CSV files in My Drive/m5_data")

store_list = ["CA_1", "TX_1", "WI_1"]
FORECAST_HORIZON, TEST_DAYS, TRAIN_WINDOW_DAYS = 7, 90, 365
SEEDS = [42, 7, 2024, 1, 99]           # 5 seeds; seed 42 reproduces the original reference DTF
T = 62

def find(name):
    hits = glob.glob(f"{OUT_DIR}/**/{name}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{name} not found in {OUT_DIR}")
    return hits[0]

# the reference DTF's own tuned settings (TPE, 21 trials per store): taken from baseline_rerun_params.csv if it exists,
# otherwise from tuning_log.csv (rows "Baseline DTF", written by the main notebook)
PKEYS = ["tweedie_variance_power", "n_estimators", "num_leaves", "min_data_in_leaf", "learning_rate",
         "feature_fraction", "max_bin", "lambda_l1", "lambda_l2"]
INT_KEYS = ["n_estimators", "num_leaves", "min_data_in_leaf", "max_bin"]
src = glob.glob(f"{OUT_DIR}/**/baseline_rerun_params.csv", recursive=True)
if src:
    bp = pd.read_csv(src[0]); print("Reference settings from baseline_rerun_params.csv")
else:
    bp = pd.read_csv(find("tuning_log.csv")); bp = bp[bp["method"] == "Baseline DTF"]
    print("Reference settings from tuning_log.csv (Baseline DTF rows)")
REF_PARAMS = {}
for s in store_list:
    rows = bp[bp["group"] == s]
    assert len(rows), f"no reference settings found for {s}"
    r = rows.iloc[-1]
    REF_PARAMS[s] = {"objective": "tweedie", **{k: (int(r[k]) if k in INT_KEYS else float(r[k])) for k in PKEYS}}
    print(s, REF_PARAMS[s])

## Step 1 - Data and inputs of the reference DTF (same code as the reference DTF notebook)

In [ ]:
cal = pd.read_csv(f"{DATA_DIR}/calendar.csv", parse_dates=["date"])
if "d" not in cal.columns:
    cal["d"] = ["d_" + str(i + 1) for i in range(len(cal))]
prices = pd.read_csv(f"{DATA_DIR}/sell_prices.csv")
sales_path = f"{DATA_DIR}/sales_train_evaluation.csv"
id_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in pd.read_csv(sales_path, nrows=0).columns if c.startswith("d_")]
snap_cols = {"CA": "snap_CA", "TX": "snap_TX", "WI": "snap_WI"}
max_date = cal["date"].max()
test_start = max_date - pd.Timedelta(days=TEST_DAYS - 1)

LAGS = list(range(1, 15)); WINDOWS = [7, 14, 30, 60, 180]
MANUAL_FEATURES = ([f"sales_lag_{l}" for l in LAGS] + [f"rolling_mean_{w}" for w in WINDOWS] + [f"rolling_stdv_{w}" for w in WINDOWS]
                   + ["day_of_week", "day_of_month", "week_of_year", "month", "quarter", "end_of_week",
                      "sell_price", "price_ratio_to_mean", "price_ratio_to_max", "snap", "is_event"])
CAT_FEATURES = ["dept_id", "cat_id"]
KEY = ["store_id", "item_id"]
FIXED_PARAMS = {"subsample": 0.5, "subsample_freq": 1, "verbosity": -1, "n_jobs": -1}

def build_features(df, horizon=FORECAST_HORIZON):
    df = df.sort_values(["item_id", "date"]).reset_index(drop=True)
    g = df.groupby("item_id", sort=False)["sales"]
    for l in LAGS:
        df[f"sales_lag_{l}"] = g.shift(l).astype("float32")
    df["_shifted"] = g.shift(horizon)
    gs = df.groupby("item_id", sort=False)["_shifted"]
    for w in WINDOWS:
        df[f"rolling_mean_{w}"] = gs.rolling(w).mean().reset_index(level=0, drop=True).astype("float32")
        df[f"rolling_stdv_{w}"] = gs.rolling(w).std().reset_index(level=0, drop=True).astype("float32")
    df = df.drop(columns="_shifted")
    df["day_of_week"] = df["date"].dt.dayofweek
    df["day_of_month"] = df["date"].dt.day
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
    df["month"] = df["date"].dt.month
    df["quarter"] = df["date"].dt.quarter
    df["end_of_week"] = (df["day_of_week"] >= 5).astype(int)
    df["is_event"] = df["event_name_1"].notna().astype(int)
    gp = df.groupby("item_id", sort=False)["sell_price"]
    csum = df["sell_price"].fillna(0).groupby(df["item_id"], sort=False).cumsum()
    ccnt = df["sell_price"].notna().groupby(df["item_id"], sort=False).cumsum()
    df["price_ratio_to_mean"] = (df["sell_price"] / (csum / ccnt.replace(0, np.nan))).astype("float32")
    df["price_ratio_to_max"] = (df["sell_price"] / gp.cummax()).astype("float32")
    return df

def to_category(df):
    df = df.copy()
    for c in CAT_FEATURES:
        df[c] = df[c].astype("category")
    return df

for s in store_list:
    trp, tep = f"{FEATS_DIR}/{s}_train.parquet", f"{FEATS_DIR}/{s}_test.parquet"
    if os.path.exists(trp) and os.path.exists(tep):
        print(f"{s}: inputs ready"); continue
    if os.path.exists(f"{STEP_DIR}/ref5_{s}.parquet"):
        print(f"{s}: forecasts already saved, inputs not needed"); continue
    parts = [ch[ch["store_id"] == s] for ch in pd.read_csv(sales_path, chunksize=2000)]
    st_ = pd.concat(parts, ignore_index=True); del parts
    state = st_["state_id"].iloc[0]
    long = st_.melt(id_vars=id_cols, value_vars=day_cols, var_name="d", value_name="sales"); del st_
    long["sales"] = long["sales"].astype("float32")
    long = long.merge(cal[["d", "date", "wm_yr_wk", "wday", "event_name_1", "event_type_1", snap_cols[state]]], on="d", how="left") \
               .rename(columns={snap_cols[state]: "snap"})
    long = long.merge(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    long = long[long["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS + 200)]
    f = build_features(long); del long; gc.collect()
    f[(f["date"] < test_start) & (f["date"] >= test_start - pd.Timedelta(days=TRAIN_WINDOW_DAYS))].to_parquet(trp, index=False)
    f[f["date"] >= test_start].to_parquet(tep, index=False)
    print(f"{s}: inputs built"); del f; gc.collect()

## Step 2 - C1: reference DTF averaged over 5 seeds (its own tuned settings, no new tuning)

In [ ]:
for s in store_list:
    path = f"{STEP_DIR}/ref5_{s}.parquet"
    if os.path.exists(path):
        print(f"{s}: already done"); continue
    t0 = time.time()
    tr = to_category(pd.read_parquet(f"{FEATS_DIR}/{s}_train.parquet").dropna(subset=MANUAL_FEATURES + ["sales"]))
    te = to_category(pd.read_parquet(f"{FEATS_DIR}/{s}_test.parquet").dropna(subset=MANUAL_FEATURES))
    out = te[KEY + ["date", "sales"]].copy()
    for seed in SEEDS:
        m = lgb.LGBMRegressor(**REF_PARAMS[s], **FIXED_PARAMS, seed=seed)
        m.fit(tr[MANUAL_FEATURES + CAT_FEATURES], tr["sales"], categorical_feature=CAT_FEATURES)
        out[f"pred_{seed}"] = m.predict(te[MANUAL_FEATURES + CAT_FEATURES]); del m; gc.collect()
    out.to_parquet(path, index=False)
    print(f"{s}: 5 seeds done ({(time.time() - t0) / 60:.1f} min)"); del tr, te, out; gc.collect()

ref5 = pd.concat([pd.read_parquet(f"{STEP_DIR}/ref5_{s}.parquet") for s in store_list], ignore_index=True)
ref5["pred"] = ref5[[f"pred_{x}" for x in SEEDS]].clip(lower=0).mean(axis=1)
ref5[KEY + ["date", "sales", "pred"]].assign(method="Reference DTF, 5 seeds").to_csv(f"{OUT_DIR}/reference5_predictions.csv.gz", index=False)
ref1 = ref5[KEY + ["date", "sales"]].assign(pred=ref5["pred_42"])     # seed 42 alone (reproduction check)

## Step 3 - Evaluation (same calculations as the final-results notebook)

In [ ]:
pp = pd.read_csv(find("eval_products.csv"))
sales = pd.concat([ch[ch["store_id"].isin(store_list)] for ch in pd.read_csv(sales_path, chunksize=3000)])
sales = pp[["store_id", "item_id"]].merge(sales, on=["store_id", "item_id"], how="left")
dcols = [c for c in sales.columns if c.startswith("d_")]
Y = sales[dcols].values.astype(float); N = len(pp)
T0 = int(np.where(cal["date"].values[:len(dcols)] == np.datetime64("2016-03-22"))[0][0])
A = Y[:, T0:T0 + T]

def to_mat(p):
    p = pp[["store_id", "item_id"]].reset_index().merge(p, on=["store_id", "item_id"]).sort_values(["index", "date"])
    p = p[pd.to_datetime(p["date"]) < test_start + pd.Timedelta(days=T)]
    assert len(p) == N * T, f"{len(p)} rows, expected {N * T}"
    assert np.allclose(p["sales"].values.reshape(N, T), A), "actuals do not match"
    return np.maximum(p["pred"].values.reshape(N, T), 0)
def load_pred(fname):
    return to_mat(pd.read_csv(find(fname), parse_dates=["date"]))

F = {"Reference DTF": load_pred("baseline_predictions.csv.gz"),
     "Reference DTF (seed 42, re-run)": to_mat(ref1),
     "Reference DTF (5 seeds)": to_mat(ref5[KEY + ["date", "sales", "pred"]])}
MEMBERS = ["optimization4_predictions.csv.gz", "optimization5_predictions.csv.gz", "optimization6_predictions.csv.gz",
           "optimization7_predictions.csv.gz", "optimization10_predictions.csv.gz"]
F["TSO-DTF"] = sum(load_pred(f) for f in MEMBERS) / 5

w = pp["weight"].values
first = np.argmax(Y[:, :T0] > 0, axis=1)
scale = np.array([np.mean(np.diff(Y[i, first[i]:T0]) ** 2) for i in range(N)])
mad = np.mean(np.abs(np.diff(Y[:, T0 - 365:T0], axis=1)), axis=1)
wk = cal["wm_yr_wk"].values[:len(dcols)]
pm = prices[prices["store_id"].isin(store_list)].pivot_table(index=["store_id", "item_id"], columns="wm_yr_wk", values="sell_price") \
        .reindex(pd.MultiIndex.from_frame(pp[["store_id", "item_id"]]))
rev28 = np.nansum(Y[:, T0 - 28:T0] * np.nan_to_num(pm.reindex(columns=wk[T0 - 28:T0]).values), axis=1)
dept = np.array([i.rsplit("_", 1)[0] for i in pp.item_id]); catg = np.array([i.split("_")[0] for i in pp.item_id]); st = pp.store_id.values
LEVELS = {"Total": np.zeros(N, int), "Store": st, "Category": catg, "Department": dept, "Store-category": np.char.add(st, catg),
          "Store-department": np.char.add(st, dept), "Product (all stores)": pp.item_id.values, "Product-store": np.arange(N)}
OFFICIAL_12 = ["Total", "Store", "Store", "Category", "Department", "Store-category", "Store-category",
               "Store-department", "Store-department", "Product (all stores)", "Product-store", "Product-store"]
PRE = {}
for lv, g in LEVELS.items():
    codes = pd.Series(g).astype("category").cat.codes.values; K = codes.max() + 1
    agg = lambda M, codes=codes, K=K: np.vstack([np.bincount(codes, weights=M[:, t], minlength=K) for t in range(M.shape[1])]).T
    Yh = agg(Y[:, :T0]); sc = np.array([np.mean(np.diff(y[np.argmax(y > 0):]) ** 2) if (y > 0).sum() > 1 else 0 for y in Yh])
    wl = np.bincount(codes, weights=rev28, minlength=K); ok = sc > 0
    PRE[lv] = (agg, agg(A)[ok], sc[ok], wl[ok] / wl[ok].sum(), ok)
def level_wrmsse(P, lv):
    agg, Aa, sc, wl, ok = PRE[lv]; Pa = agg(P)[ok]
    return (wl * np.sqrt(np.mean((Aa - Pa) ** 2, axis=1) / sc)).sum()
def official(P):
    return np.mean([level_wrmsse(P, lv) for lv in OFFICIAL_12])
def official_daily(P):
    """Official-score loss for each day: mean over the 12 levels of the weighted squared scaled error of that day."""
    out = np.zeros(T)
    for lv in OFFICIAL_12:
        agg, Aa, sc, wl, ok = PRE[lv]; Pa = agg(P)[ok]
        out += (wl[:, None] * (Aa - Pa) ** 2 / sc[:, None]).sum(axis=0) / len(OFFICIAL_12)
    return out
def product_metrics(P):
    e = A - P; r = np.sqrt(np.mean(e ** 2, axis=1) / scale)
    return {"WRMSSE (product level)": (w * r).sum() / w.sum(), "RMSSE mean": r.mean(), "RMSSE median": np.median(r),
            "RMSE": np.sqrt((e ** 2).mean()), "MAE": np.abs(e).mean(), "WAPE": np.abs(e).sum() / A.sum(),
            "|SME| bias": abs(np.nanmean(np.mean(e, axis=1) / mad))}, r
def product_daily(P):
    return ((w / w.sum())[:, None] * (A - P) ** 2 / scale[:, None]).sum(axis=0)

rows, R = [], {}
for k, P in F.items():
    m, r = product_metrics(P); R[k] = r
    rows.append({"Method": k, "Official M5 score": official(P), **m})
tab = pd.DataFrame(rows).set_index("Method")
print(tab.round(4).to_string())
chk = abs(tab.loc["Reference DTF (seed 42, re-run)", "Official M5 score"] - tab.loc["Reference DTF", "Official M5 score"])
print(f"\nReproduction check (seed 42 re-run vs saved reference): official score difference {chk:.5f}")

imp = lambda a, b: 100 * (a - b) / a
c1 = pd.DataFrame({"vs Reference DTF (1 model) %": imp(tab.loc["Reference DTF"], tab.loc["TSO-DTF"]),
                   "vs Reference DTF (5 seeds) %": imp(tab.loc["Reference DTF (5 seeds)"], tab.loc["TSO-DTF"]),
                   "5 seeds vs 1 model %": imp(tab.loc["Reference DTF"], tab.loc["Reference DTF (5 seeds)"])})
print("\nC1 - improvement of TSO-DTF over each reference:\n" + c1.round(2).to_string())

## Step 4 - C3: Diebold-Mariano tests and effect sizes

In [ ]:
def dm_test(loss_ref, loss_new, lag=6):
    """DM test with Newey-West variance (lag days) and the Harvey-Leybourne-Newbold correction. One-sided: new better."""
    d = loss_ref - loss_new; n = len(d); dbar = d.mean(); dc = d - dbar
    gam = [np.sum(dc[k:] * dc[:n - k]) / n for k in range(lag + 1)]
    var = gam[0] + 2 * sum((1 - k / (lag + 1)) * gam[k] for k in range(1, lag + 1))
    dm = dbar / np.sqrt(var / n)
    h = 1; hln = np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)
    stat = dm * hln
    return stat, 1 - stats.t.cdf(stat, df=n - 1)

dm_rows = []
for ref in ["Reference DTF", "Reference DTF (5 seeds)"]:
    for name, fn in [("Official M5 (daily loss)", official_daily), ("Product-level WRMSSE (daily loss)", product_daily)]:
        lr, ln = fn(F[ref]), fn(F["TSO-DTF"])
        stat, p = dm_test(lr, ln)
        dm_rows.append({"Comparison": f"TSO-DTF vs {ref}", "Loss": name, "DM statistic": stat, "p-value (one-sided)": p,
                        "Days TSO-DTF better": int((ln < lr).sum()), "of days": T})
dm = pd.DataFrame(dm_rows)
print(dm.round(4).to_string(index=False))

eff_rows = []
for ref in ["Reference DTF", "Reference DTF (5 seeds)"]:
    rb, rn = R[ref], R["TSO-DTF"]; diff = rb - rn
    ok = np.isfinite(diff) & (diff != 0)
    ranks = stats.rankdata(np.abs(diff[ok])); wplus = ranks[diff[ok] > 0].sum(); wminus = ranks[diff[ok] < 0].sum()
    eff_rows.append({"Comparison": f"TSO-DTF vs {ref}",
                     "Products improved %": 100 * np.mean(rn < rb),
                     "Median per-product RMSSE improvement %": 100 * np.nanmedian(diff / rb),
                     "Mean per-product RMSSE improvement %": 100 * np.nanmean(diff / rb),
                     "Rank-biserial correlation": (wplus - wminus) / (wplus + wminus),
                     "Wilcoxon p": stats.wilcoxon(rn, rb).pvalue})
eff = pd.DataFrame(eff_rows)
print("\n" + eff.round(4).to_string(index=False))

tab.to_csv(f"{OUT_DIR}/extra_checks_metrics.csv"); c1.to_csv(f"{OUT_DIR}/extra_checks_c1.csv")
dm.to_csv(f"{OUT_DIR}/extra_checks_dm.csv", index=False); eff.to_csv(f"{OUT_DIR}/extra_checks_effect.csv", index=False)

print("\n================ SUMMARY ================")
print(f"C1: TSO-DTF vs reference (1 model):  official {c1.iloc[0, 0]:.2f}% | product {c1.iloc[1, 0]:.2f}%")
print(f"C1: TSO-DTF vs reference (5 seeds):  official {c1.iloc[0, 1]:.2f}% | product {c1.iloc[1, 1]:.2f}%")
print(f"C1: TSO-DTF better than the 5-seed reference on every metric: {'YES' if (c1.iloc[:, 1] > 0).all() else 'NO'}")
for _, r in dm.iterrows():
    print(f"C3: {r['Comparison']}, {r['Loss']}: DM = {r['DM statistic']:.2f}, p = {r['p-value (one-sided)']:.4f}")

## Step 5 - Download the results (send extra_checks_results.zip to the chat)

In [ ]:
zdir = f"{OUT_DIR}/_extrazip"; os.makedirs(zdir, exist_ok=True)
for f in ["extra_checks_metrics.csv", "extra_checks_c1.csv", "extra_checks_dm.csv", "extra_checks_effect.csv"]:
    shutil.copy(f"{OUT_DIR}/{f}", zdir)
zpath = shutil.make_archive("/content/extra_checks_results" if IN_COLAB else f"{WORK_DIR}/extra_checks_results", "zip", zdir); shutil.rmtree(zdir)
print("Saved:", zpath)
if IN_COLAB:
    from google.colab import files
    files.download(zpath)